# KarmaQuiz demo (Team PromptPilot, PE-B37-07)

Runs the full prompt chain on a public passage: **T1 generate → automatic checks → T2 review (⇄ T1R revise) → T3 tag**, then **T4 gap summary** on synthetic learner answers.

The outputs below were saved from a real run. To run it again in Colab, add your Gemini key as a Colab secret named `GOOGLE_API_KEY` (key icon on the left), then choose *Runtime → Run all*.

In [1]:
import os, sys, pathlib, json
if 'google.colab' in sys.modules:
    !git clone -q https://github.com/varun-sharma-2006/PromptPilot.git 2>/dev/null || true
    !pip -q install google-genai
    from google.colab import userdata
    os.environ['GOOGLE_API_KEY'] = userdata.get('GOOGLE_API_KEY')
    ROOT = pathlib.Path('PromptPilot')
else:
    ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'code' else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT / 'code'))
import chain, llm
print('Model:', llm.MODEL)

Model: gemini-3.1-flash-lite


In [2]:
p = json.loads((ROOT / 'test-data/passages/P2_gem.json').read_text(encoding='utf-8'))
print(p['title'], '|', p['source'], '|', p['licence'])
print(p['text'][:600], '...')

Government e Marketplace | https://en.wikipedia.org/w/index.php?oldid=1386391530 | CC BY-SA 4.0 (Wikipedia)
Government e Marketplace (GeM) is an online platform for public procurement in India. The initiative was launched on 9 August 2016, by the Ministry of Commerce and Industry in the Government of India, with the objective to create an open and transparent procurement platform for government buyers. It was built in a record time of 5 months to facilitate the online procurement of goods and Services. Use of GeM by Government users has been authorized and made mandatory by the Ministry of Finance by adding a new Rule No. 149 in the General Financial Rules, 2017.
The platform is owned by GeM SPV, a  ...


In [3]:
c = chain.Chain()
result = c.run(p['text'], p['competency'], 'medium', 3)
for rnd in result['review_rounds']:
    print(f"T2 round {rnd['round']}:", [r.get('verdict') for r in rnd['reviews']])
print('Generator note:', result['note'] or '(none)')
for i, q in enumerate(result['questions'], 1):
    print(f"\nQ{i}. {q['question']}")
    for k in 'ABCD':
        print(f"   {k}. {q['options'][k]}{'   <-- key' if k == q['correct'] else ''}")
    print('   Source:', q['source_sentence'])
    print('   Tag:', q['competency'], '|', q['tagged_difficulty'], '|', q['tag_justification'])

T1: generating questions


Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


T2: reviewing 2 question(s), round 1


T3: tagging 2 accepted question(s)


T2 round 1: ['pass', 'pass']
Generator note: (none)

Q1. How has the Government of India ensured that public procurement is conducted through the GeM platform?
   A. By transferring all procurement authority to the Ministry of Commerce
   B. By dissolving the DGS&D and replacing it with a private entity
   C. By offering financial incentives to departments that use the portal
   D. By mandating its use through an amendment to the General Financial Rules   <-- key
   Source: Use of GeM by Government users has been authorized and made mandatory by the Ministry of Finance by adding a new Rule No. 149 in the General Financial Rules, 2017.
   Tag: Functional: Public Procurement | easy | The question tests factual recall regarding the legal mandate for GeM usage under General Financial Rules.

Q2. In what way has the implementation of GeM altered the previous landscape of government procurement?
   A. It replaced fragmented manual processes with a system focused on openness and fairness   <-

In [4]:
# Synthetic learner: answers the first question correctly and the rest wrongly
qs = result['questions']
wrong = {'A': 'B', 'B': 'C', 'C': 'D', 'D': 'A'}
answers = [q['correct'] if i == 0 else wrong[q['correct']] for i, q in enumerate(qs)]
gaps = c.gap_summary(qs, answers)
for row in gaps['table']:
    print(f"{row['competency']}: {row['correct']}/{row['total']}")
print(json.dumps(gaps['summary'], indent=2, ensure_ascii=False))

Functional: Public Procurement: 1/2
{
  "overall": "You answered 1 out of 2 questions correctly on this quiz.",
  "gaps": [
    {
      "competency": "Functional: Public Procurement",
      "level": "partial",
      "what_to_review": "Review how the implementation of GeM has altered the previous landscape of government procurement."
    }
  ],
  "next_step": "Start by reviewing the specific impact of GeM on government procurement processes to strengthen your understanding."
}


In [5]:
print(f'{len(c.log)} LLM calls in this run:')
for e in c.log:
    print(' -', e['prompt'], e['meta'])

4 LLM calls in this run:
 - final/T1_generate {'model': 'gemini-3.1-flash-lite', 'temperature': 0.2, 'json_mode': True, 'seconds': 45.9}
 - final/T2_review {'model': 'gemini-3.1-flash-lite', 'temperature': 0.0, 'json_mode': True, 'seconds': 1.4}
 - final/T3_tag {'model': 'gemini-3.1-flash-lite', 'temperature': 0.0, 'json_mode': True, 'seconds': 84.4}
 - final/T4_gaps {'model': 'gemini-3.1-flash-lite', 'temperature': 0.2, 'json_mode': True, 'seconds': 2.3}
